In [40]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import pystac
from dotenv import load_dotenv
load_dotenv()
import boto3
import os
import odc.stac
from typing import Tuple
import xml.etree.ElementTree as ET
from pyproj import Transformer
import xarray as xr
import glob

In [41]:
ENDPOINT_URL = "https://eodata.dataspace.copernicus.eu"
BUCKET_NAME = "eodata"
os.environ["AWS_ACCESS_KEY_ID"] = os.environ["CDSE_S3_ACCESS_KEY"]
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ["CDSE_S3_SECRET_KEY"]
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ.pop("AWS_REGION", None)
os.environ.pop("AWS_NO_SIGN_REQUEST", None)

In [42]:
DATA = Path("../data")

In [43]:
os.makedirs(DATA / "sample_captures", exist_ok=True)

In [44]:
matchups = gpd.read_file(DATA / "registros_limpios_matchups.json")

In [45]:
ic = pystac.ItemCollection.from_file(str(DATA/ "matchup_items.json"))
item_by_id = {item.id : item for item in ic}

In [46]:
matchups.head(2)

,fecha,fuente,chla,grupo_nombre,estado_trofico,delta,pass_id,geometry
0,2017-01-02,OAN,8.9,RDP-MONTES,M,2 days 13:50:52.026000,S2A_MSIL1C_20170104T135052_N0500_R024_T21HUC_2...,POINT (401164.986 6214585.025)
1,2017-01-02,GEMS,6.8,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,POINT (678380.002 6147956.963)


In [47]:
to_wgs84 = Transformer.from_crs("EPSG:32721", "EPSG:4326", always_xy=True)

In [48]:
BAND_RESOLUTION = { "B01": 60, "B02": 10, "B03": 10, "B04": 10, "B05": 20, "B06": 20, "B07": 20, "B08": 10, "B8A": 20, "B09": 60, "B10": 60, "B11": 20, "B12": 20}
def bands_downsample(bands: list[str]):
    res = -float("inf")
    for b in bands:
        res = max(res, BAND_RESOLUTION[b])
    return res

In [49]:
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=os.environ["CDSE_S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["CDSE_S3_SECRET_KEY"],
)

In [10]:
def item_quant_and_offset(item: pystac.item.Item) -> Tuple[float, float]:
    s3_key = item.assets["product_metadata"].href.replace("s3://eodata/", "")
    xml_bytes = s3.get_object(Bucket=BUCKET_NAME, Key=s3_key)["Body"].read()
    
    # Descargamos la metadata (en formato en XML)
    parsed_xml = ET.fromstring(xml_bytes)
    quantification = float(parsed_xml.find(".//QUANTIFICATION_VALUE").text)
    radio_add_offset = float(parsed_xml.find(".//RADIO_ADD_OFFSET").text)
    
    return quantification, radio_add_offset

In [11]:
type(ic[0])

pystac.item.Item

In [12]:
#item_quant_and_offset(ic[0])

(10000.0, -1000.0)

In [13]:
BANDS = [
    "B07",
    "B06",
    "B05",
    "B04",
    "B03",
    "B02"
]
RES = bands_downsample(BANDS)

In [14]:
m = matchups.loc[0]
x, y = m.geometry.x, m.geometry.y

x, y = to_wgs84.transform(x, y)

d = 0.005
       # xmin , ymin, xmax, ymax
bbox = [x - d, y - d, x + d, y + d]

id_1 = m.pass_id

item_1 = item_by_id[id_1]

In [15]:
ds = odc.stac.load(
    [item_1],
    bands=BANDS,
    resolution=RES,
    bbox=bbox,
    chunks=None,
    resampling="average"
)

In [16]:
#1000/20

In [37]:
ds[

<xarray.Dataset> Size: 34kB
Dimensions:      (y: 57, x: 48, time: 1)
Coordinates:
  * y            (y) float64 456B 6.215e+06 6.215e+06 ... 6.214e+06 6.214e+06
  * x            (x) float64 384B 4.007e+05 4.007e+05 ... 4.016e+05 4.016e+05
  * time         (time) datetime64[us] 8B 2017-01-04T13:50:52.026000
    spatial_ref  int32 4B 32721
Data variables:
    B07          (time, y, x) uint16 5kB 7526 7498 7488 7483 ... 7458 7447 7450
    B06          (time, y, x) uint16 5kB 7329 7317 7298 7291 ... 7264 7271 7243
    B05          (time, y, x) uint16 5kB 7219 7205 7178 7143 ... 7142 7147 7162
    B04          (time, y, x) uint16 5kB 7217 7233 7207 7220 ... 7161 7140 7151
    B03          (time, y, x) uint16 5kB 7051 7031 7031 7055 ... 6938 6959 6946
    B02          (time, y, x) uint16 5kB 7443 7425 7374 7368 ... 7245 7255 7230

In [39]:
ds["B05"]

<xarray.DataArray 'B05' (time: 1, y: 57, x: 48)> Size: 5kB
array([[[7219, 7205, 7178, ..., 6964, 6917, 6910],
        [7215, 7173, 7160, ..., 6957, 6922, 6900],
        [7196, 7167, 7140, ..., 6963, 6921, 6907],
        ...,
        [7286, 7311, 7304, ..., 7175, 7144, 7144],
        [7278, 7285, 7282, ..., 7168, 7155, 7134],
        [7289, 7283, 7295, ..., 7142, 7147, 7162]]],
      shape=(1, 57, 48), dtype=uint16)
Coordinates:
  * time         (time) datetime64[us] 8B 2017-01-04T13:50:52.026000
  * y            (y) float64 456B 6.215e+06 6.215e+06 ... 6.214e+06 6.214e+06
  * x            (x) float64 384B 4.007e+05 4.007e+05 ... 4.016e+05 4.016e+05
    spatial_ref  int32 4B 32721
Attributes:
    nodata:   0

In [36]:
(ds.nbytes * 3000) / 1024**2

96.37069702148438